### Final Model: Stacking Ensemble

This notebook constructs the final classification model by stacking the three optimized gradient boosting classifiers. A Logistic Regression meta-classifier is trained on the base models' predictions to generate the final prediction. The ensemble is then evaluated on the unseen test set and compared with the best individual models.

In [1]:
import pandas as pd
import seaborn as sns 
import numpy as np 
import matplotlib.pyplot as plt 

from sklearn.model_selection import (
                            train_test_split,
                            StratifiedKFold,
                            cross_val_score,
                            cross_validate,
                            GridSearchCV,
                            RandomizedSearchCV
                            )
from sklearn.metrics import (classification_report ,
                            confusion_matrix,
                            roc_curve,
                            accuracy_score,
                            f1_score,
                            precision_recall_curve,
                            precision_score, recall_score)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
import optuna

import time
import joblib 
import wandb as wb 
from dotenv import load_dotenv, find_dotenv
import os

sns.set_theme('paper')
sns.set_style('ticks')

import warnings 
warnings.filterwarnings("ignore",    category=DeprecationWarning,
    module="wandb.analytics.sentry")

/home/yassine/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### **01. Data preprocessing**

In [2]:
df = pd.read_csv("../Data/BEED_Data.csv")
df.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,y
0,4,7,18,25,28,27,20,10,-10,-18,-20,-16,13,32,12,10,0
1,87,114,120,106,76,54,28,5,-19,-49,-85,-102,-100,-89,-61,-21,0
2,-131,-133,-140,-131,-123,-108,-58,-51,-70,-77,-76,-76,-73,-57,-40,-14,0
3,68,104,73,34,-12,-26,-38,-36,-67,-88,-25,31,18,-4,6,-29,0
4,-67,-90,-97,-94,-86,-71,-43,-11,23,46,58,50,39,19,-9,-41,0


In [3]:
# splitting the data 
X, y = df.iloc[:,:-1],df.iloc[:,-1]
X, y

(       X1   X2   X3   X4   X5   X6  X7  X8  X9  X10  X11  X12  X13  X14  X15  \
 0       4    7   18   25   28   27  20  10 -10  -18  -20  -16   13   32   12   
 1      87  114  120  106   76   54  28   5 -19  -49  -85 -102 -100  -89  -61   
 2    -131 -133 -140 -131 -123 -108 -58 -51 -70  -77  -76  -76  -73  -57  -40   
 3      68  104   73   34  -12  -26 -38 -36 -67  -88  -25   31   18   -4    6   
 4     -67  -90  -97  -94  -86  -71 -43 -11  23   46   58   50   39   19   -9   
 ...   ...  ...  ...  ...  ...  ...  ..  ..  ..  ...  ...  ...  ...  ...  ...   
 7995   -9   -4   -7    2   -9  -14  -7  12  -5   -3   -2   -2  -12   -5    4   
 7996   -8   -4   -7    1   -8  -14  -7  12  -6   -3   -2   -3  -15   -4    4   
 7997   -6   -5   -7    1   -8  -14  -7  12  -7   -3   -2   -4  -16   -4    3   
 7998   -5   -6   -6    1   -8  -13  -7  13  -7   -2   -3   -5  -15   -4    3   
 7999   -4   -7   -5    2   -8  -12  -7  14  -6    0   -3   -6  -13   -4    2   
 
       X16  
 0      10  


In [4]:
# splitting the data to training and testing sets 

x_train , x_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=42)
x_train.shape , x_test.shape , y_train.shape , y_test.shape

((5600, 16), (2400, 16), (5600,), (2400,))

In [5]:
# loading the best params 


api = wb.Api()
params_dict = {}
models = ["CAT","LGBM","XGB"]

for model_name in models:
    artifact = api.artifact(
        f"yassinedatascientist001-datalab/Epeliptic_seizure_classification/{model_name}_v1:latest" )
    params_dict[model_name] = artifact.metadata["best_params"]

print(params_dict)  


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/yassine/.netrc.


{'CAT': {'depth': 9, 'iterations': 684, 'l2_leaf_reg': 4.804799903938045, 'border_count': 144, 'learning_rate': 0.1800447538836751, 'random_strength': 3.4880900619265116, 'bagging_temperature': 0.08940761500641747}, 'LGBM': {'max_depth': 10, 'reg_alpha': 0.04042106713481747, 'subsample': 0.6498728957151503, 'num_leaves': 89, 'reg_lambda': 3.0739563781844126, 'n_estimators': 418, 'learning_rate': 0.19789626311951297, 'colsample_bytree': 0.7126250919558335, 'min_child_samples': 35}, 'XGB': {'gamma': 0.0670195965670759, 'max_depth': 4, 'reg_alpha': 0.7204956148413739, 'subsample': 0.8668673681524485, 'reg_lambda': 8.53272363359725, 'n_estimators': 588, 'learning_rate': 0.2056264000098812, 'colsample_bytree': 0.7517231204542465, 'min_child_weight': 8}}


In [6]:
params_dict

{'CAT': {'depth': 9,
  'iterations': 684,
  'l2_leaf_reg': 4.804799903938045,
  'border_count': 144,
  'learning_rate': 0.1800447538836751,
  'random_strength': 3.4880900619265116,
  'bagging_temperature': 0.08940761500641747},
 'LGBM': {'max_depth': 10,
  'reg_alpha': 0.04042106713481747,
  'subsample': 0.6498728957151503,
  'num_leaves': 89,
  'reg_lambda': 3.0739563781844126,
  'n_estimators': 418,
  'learning_rate': 0.19789626311951297,
  'colsample_bytree': 0.7126250919558335,
  'min_child_samples': 35},
 'XGB': {'gamma': 0.0670195965670759,
  'max_depth': 4,
  'reg_alpha': 0.7204956148413739,
  'subsample': 0.8668673681524485,
  'reg_lambda': 8.53272363359725,
  'n_estimators': 588,
  'learning_rate': 0.2056264000098812,
  'colsample_bytree': 0.7517231204542465,
  'min_child_weight': 8}}

In [7]:

xgb = XGBClassifier(
    random_state=42,
    **params_dict["XGB"]
)

lgbm = LGBMClassifier(
    random_state=42,
    **params_dict["LGBM"]
)

cat = CatBoostClassifier(
    random_state=42,
    verbose=False,
    **params_dict["CAT"]
)

In [ ]:
PROJECT_NAME = "Epeliptic_seizure_classification"
stack = StackingClassifier(
    estimators=[
        ("xgb",xgb),
        ("lgbm",lgbm),
        ("cat",cat)
    ],
    final_estimator=LogisticRegression(
        max_iter=1000,        
    ),
    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)
stack.fit(x_train,y_train),
preds = stack.predict(x_test)
metrics = {
    "accuracy": accuracy_score(y_test, preds),
    "f1": f1_score(y_test, preds, average="weighted"),
    "precision": precision_score(y_test, preds, average="weighted"),
    "recall": recall_score(y_test, preds, average="weighted"),
}

run = wb.init(
    project=PROJECT_NAME,
    name="Stacking_v1",
    group="Stacking",
    job_type="ensemble"
    )

run.log(metrics)
joblib.dump(
    stack,
    "../models/stacking.joblib"
)
artifact = wb.Artifact(
    name='Stacking_v1',
    type='model',
    description="the 3 models are stucked using sklearn stucking class where a meta learner is trained ",
    metadata = metrics 
    )
artifact.add_file("../models/stacking.joblib")
run.log_artifact(artifact)
run.finish()

# evaluation 
class_report = classification_report(y_test, preds)
conf_matrix = confusion_matrix(y_test, preds)
fig = sns.heatmap(conf_matrix,cmap="Blues",annot=True,fmt="d")
print(class_report)
plt.show(fig)